In [10]:
# input
metalnet_test_file = "../../../data/dataset/test_metalnet.tsv"
mmcif_dir = "../../../data/pdb/"
# output
cleaned_dir = "./tmp/pdb_cleaned"

In [11]:
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.mmcifio import MMCIFIO, Select
import pandas as pd
import tqdm
from pathlib import Path

class AtomOnly(Select):
    def accept_atom(self, atom):
        return atom.get_parent().id[0] == " "

df = pd.read_table(metalnet_test_file)
seq_ids = set(df['seq_id'])
for seq_id in tqdm.tqdm(seq_ids):
    pdb_id = seq_id.split("_")[0]
    chain_id = seq_id.split("_")[1]
    pdb_file = Path(mmcif_dir) / f"{pdb_id}.cif"
    
    struct = MMCIFParser(QUIET=True).get_structure(seq_id, pdb_file)
    target_chain = None
    for c in struct.get_chains():
        if c.id == chain_id:
            target_chain = c

    assert target_chain is not None

    io = MMCIFIO()
    io.set_structure(target_chain)
    io.save(open(Path(cleaned_dir) / f"{pdb_id}_{chain_id}.cif", "w"), AtomOnly())


100%|██████████| 447/447 [04:25<00:00,  1.68it/s]
